# Lab 11: Projection — The Best Shadow

This lab is a full computational companion to Chapter 11.  
The goal is not only to practice formulas, but to *see* projection as best approximation.

You will explore:

- projection onto a line,
- residuals and orthogonality,
- projection matrices,
- least-squares fitting,
- signal approximation,
- image projection,
- high-dimensional projections.

Main idea:

$$
y=\hat y+r, \qquad \hat y\in S, \qquad r\perp S.
$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(11)


## 1. Projection onto a line

For a nonzero vector $u$, the projection of $y$ onto $\operatorname{span}(u)$ is

$$
\operatorname{proj}_{\operatorname{span}(u)}(y)=rac{u\cdot y}{u\cdot u}u.
$$


In [ ]:
def project_onto_line(y, u):
    y = np.asarray(y, dtype=float)
    u = np.asarray(u, dtype=float)
    if np.allclose(u, 0):
        raise ValueError("u must be nonzero")
    return (np.dot(u, y) / np.dot(u, u)) * u

y = np.array([4.0, 3.0])
u = np.array([2.0, 1.0])
proj = project_onto_line(y, u)
resid = y - proj

print("y      =", y)
print("u      =", u)
print("proj   =", proj)
print("resid  =", resid)
print("u·resid=", np.dot(u, resid))


In [ ]:
def plot_projection(y, u, ax=None, title="Projection onto a line"):
    y = np.asarray(y, dtype=float)
    u = np.asarray(u, dtype=float)
    proj = project_onto_line(y, u)
    resid = y - proj
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    ts = np.linspace(-3, 4, 200)
    line = np.outer(ts, u)
    ax.plot(line[:,0], line[:,1], label="span(u)")
    ax.quiver(0, 0, y[0], y[1], angles="xy", scale_units="xy", scale=1, label="y")
    ax.quiver(0, 0, proj[0], proj[1], angles="xy", scale_units="xy", scale=1, label="projection")
    ax.quiver(proj[0], proj[1], resid[0], resid[1], angles="xy", scale_units="xy", scale=1, label="residual")
    ax.scatter([proj[0]], [proj[1]])
    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.set_aspect("equal")
    ax.grid(True)
    ax.legend()
    ax.set_title(title)
    m = max(1, np.max(np.abs(np.r_[line.flatten(), y, proj])) + 1)
    ax.set_xlim(-m, m)
    ax.set_ylim(-m, m)
    return ax

plot_projection(y, u)
plt.show()


### Student task 1
Change `y` and `u` in the next cell.  
Before running the code, predict whether the residual should point mostly upward, downward, left, or right.

In [ ]:
y2 = np.array([1.0, 5.0])
u2 = np.array([3.0, 1.0])
plot_projection(y2, u2, title="Your projection experiment")
plt.show()
print("projection:", project_onto_line(y2, u2))
print("residual:", y2 - project_onto_line(y2, u2))
print("dot product with u:", np.dot(u2, y2 - project_onto_line(y2, u2)))


## 2. The best scalar really minimizes distance

A projection onto a line has the form $cu$.  
The projection coefficient is the value of $c$ that minimizes

$$
\|y-cu\|^2.
$$


In [ ]:
y = np.array([4.0, 3.0])
u = np.array([2.0, 1.0])
c_star = np.dot(u, y) / np.dot(u, u)
cs = np.linspace(c_star - 3, c_star + 3, 400)
errors = np.array([np.linalg.norm(y - c*u)**2 for c in cs])

plt.figure(figsize=(7, 4.5))
plt.plot(cs, errors)
plt.axvline(c_star, linestyle="--", label=f"best c = {c_star:.3f}")
plt.xlabel("c")
plt.ylabel(r"$||y-cu||^2$")
plt.title("Projection coefficient as an error minimizer")
plt.grid(True)
plt.legend()
plt.show()


## 3. Projection matrices

If $q$ is a unit vector, the projection matrix onto $\operatorname{span}(q)$ is

$$
P=qq^T.
$$

A projection matrix satisfies:

$$
P^T=P, \qquad P^2=P.
$$


In [ ]:
q = np.array([2.0, 1.0, 2.0])
q = q / np.linalg.norm(q)
P = np.outer(q, q)

print("q =", q)
print("P =")
print(P)
print("P.T - P =")
print(np.round(P.T - P, 10))
print("P@P - P =")
print(np.round(P @ P - P, 10))


## 4. Projection onto a subspace using QR

When the approximation space has several directions, QR factorization gives an orthonormal basis.
If $Q$ has orthonormal columns, then

$$
\hat y = QQ^T y.
$$


In [ ]:
B = rng.normal(size=(5, 2))
Q, R = np.linalg.qr(B)
y = rng.normal(size=5)

proj = Q @ (Q.T @ y)
resid = y - proj

print("Q.T @ Q =")
print(np.round(Q.T @ Q, 8))
print("Q.T @ residual =", np.round(Q.T @ resid, 10))
print("||y||^2:", np.dot(y, y))
print("||proj||^2 + ||resid||^2:", np.dot(proj, proj) + np.dot(resid, resid))


## 5. Least squares as projection

When $Ax=b$ cannot be solved exactly, least squares finds the vector $A\hat{x}$ in the column space of $A$ closest to $b$:

$$
\hat{x}=rg\min_x \|Ax-b\|^2.
$$

The residual satisfies

$$
A^T(b-A\hat{x})=0.
$$


In [ ]:
x_data = np.array([0, 1, 2, 3, 4, 5], dtype=float)
y_data = np.array([1.2, 1.7, 3.1, 3.8, 5.0, 5.4], dtype=float)

A = np.column_stack([np.ones_like(x_data), x_data])
b = y_data

x_hat = np.linalg.solve(A.T @ A, A.T @ b)
fitted = A @ x_hat
resid = b - fitted

print("least-squares coefficients [intercept, slope] =", x_hat)
print("A.T @ residual =", np.round(A.T @ resid, 10))

xx = np.linspace(-0.5, 5.5, 300)
yy = x_hat[0] + x_hat[1] * xx
plt.figure(figsize=(7, 5))
plt.scatter(x_data, y_data, label="data")
plt.plot(xx, yy, label="least-squares line")
for xi, yi, fi in zip(x_data, y_data, fitted):
    plt.plot([xi, xi], [fi, yi], linestyle="--", linewidth=1)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Least squares: vertical residuals")
plt.grid(True)
plt.legend()
plt.show()


### Student task 2
Add one outlier to the data and rerun the least-squares fit.  
How much does one point change the projection?

In [ ]:
x_out = np.append(x_data, 5.5)
y_out = np.append(y_data, 10.0)  # change this outlier
A_out = np.column_stack([np.ones_like(x_out), x_out])
coef_out = np.linalg.lstsq(A_out, y_out, rcond=None)[0]

plt.figure(figsize=(7, 5))
plt.scatter(x_data, y_data, label="original data")
plt.scatter([x_out[-1]], [y_out[-1]], marker="x", s=100, label="outlier")
plt.plot(xx, x_hat[0] + x_hat[1]*xx, label="original fit")
plt.plot(xx, coef_out[0] + coef_out[1]*xx, linestyle="--", label="fit with outlier")
plt.grid(True)
plt.legend()
plt.title("Sensitivity of least squares to an outlier")
plt.show()
print("original coefficients:", x_hat)
print("with outlier:", coef_out)


## 6. Projection for signal approximation

A signal is a vector. We can approximate it by projecting onto a low-dimensional space spanned by simple patterns.

In [ ]:
n = 200
t = np.linspace(0, 1, n)
signal = 1.0*np.sin(2*np.pi*t) + 0.45*np.sin(6*np.pi*t) + 0.25*rng.normal(size=n)

# basis: constant, sin(2πt), cos(2πt), sin(6πt), cos(6πt)
A = np.column_stack([
    np.ones(n),
    np.sin(2*np.pi*t), np.cos(2*np.pi*t),
    np.sin(6*np.pi*t), np.cos(6*np.pi*t)
])
coef = np.linalg.lstsq(A, signal, rcond=None)[0]
approx = A @ coef
resid = signal - approx

plt.figure(figsize=(9, 4))
plt.plot(t, signal, label="noisy signal")
plt.plot(t, approx, linewidth=3, label="projection onto signal space")
plt.plot(t, resid, linestyle="--", label="residual")
plt.grid(True)
plt.legend()
plt.title("Signal approximation by projection")
plt.show()
print("residual norm:", np.linalg.norm(resid))
print("A.T @ residual, rounded:", np.round(A.T @ resid, 8))


## 7. Image approximation by projection

A grayscale image can be stored as a vector.  
Here we create simple image-like patterns and project an image onto the span of these patterns.

In [ ]:
m = 32
x = np.linspace(-1, 1, m)
X, Y = np.meshgrid(x, x)
img = np.exp(-8*((X-0.25)**2 + (Y+0.1)**2)) + 0.6*np.exp(-15*((X+0.35)**2 + (Y-0.25)**2))
img += 0.08*rng.normal(size=(m, m))

basis_imgs = [
    np.ones((m,m)),
    X,
    Y,
    X*Y,
    X**2,
    Y**2,
]
A = np.column_stack([B.reshape(-1) for B in basis_imgs])
b = img.reshape(-1)
coef = np.linalg.lstsq(A, b, rcond=None)[0]
approx = (A @ coef).reshape(m, m)
resid = img - approx

for title, arr in [("image", img), ("projection", approx), ("residual", resid)]:
    plt.figure(figsize=(4,4))
    plt.imshow(arr, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()
print("relative residual:", np.linalg.norm(resid)/np.linalg.norm(img))


## 8. High-dimensional projection

In high dimensions, projecting onto a $k$-dimensional subspace keeps roughly a fraction $k/n$ of the squared length of a random vector. This is a powerful geometric intuition behind dimension reduction.

In [ ]:
n = 200
trials = 1000
ks = [1, 2, 5, 10, 20, 50, 100]
fractions = []
for k in ks:
    vals = []
    B = rng.normal(size=(n, k))
    Q, _ = np.linalg.qr(B)
    for _ in range(trials):
        y = rng.normal(size=n)
        proj = Q @ (Q.T @ y)
        vals.append(np.dot(proj, proj) / np.dot(y, y))
    fractions.append(vals)

plt.figure(figsize=(8, 4.8))
plt.boxplot(fractions, labels=[str(k) for k in ks])
plt.plot(range(1, len(ks)+1), [k/n for k in ks], marker="o", label="k/n")
plt.xlabel("subspace dimension k")
plt.ylabel("fraction of squared length kept")
plt.title("Random high-dimensional projections")
plt.grid(True)
plt.legend()
plt.show()


## 9. Final reflection

Write answers in your own words.

1. What does projection mean geometrically?
2. Why is the residual perpendicular to the approximation space?
3. How is least squares a projection problem?
4. In a data science model, what is the meaning of the residual?
5. How does projection prepare the way for PCA and compression?
